In [ ]:
import numpy as np
import pandas as pd
import random as rnd
from matplotlib import pyplot as plt
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
import shap
import optuna
import math
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.metrics import roc_curve, precision_recall_curve, f1_score, auc
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    log_loss
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
def load_dataset(filename):
    data_train = pd.read_csv(filename,sep=",", names = [
    'fLength', 'fWidth', 'fSize', 'fConc', 'fConc1',
    'fAsym', 'fM3Long', 'fM3Trans', 'fAlpha', 'fDist', 'class'
    ])
    #permute the data
    data_train = data_train.sample(frac=1).reset_index(drop=True) # shuffle the data
    X = data_train.iloc[:, 0:10].values # Get first two columns as the input
    Y = data_train.iloc[:, 10].values # Get the third column as the label
    Y[Y=="g"]=1
    Y[Y=="h"]=0
    Y = Y.astype(int)
    return X,Y



# Load the dataset
X, Y = load_dataset("https://archive.ics.uci.edu/ml/machine-learning-databases/magic/magic04.data")


# Division between test and train dataset
rng = rnd.Random(42)
idx = list(range(X.shape[0]))
test_idx = rng.sample(idx, int(X.shape[0]*0.25))
train_idx = [i for i in idx if i not in test_idx]
X_training = X[train_idx]
X_test  = X[test_idx]
Y_training = Y[train_idx]
Y_test  = Y[test_idx]

# Validation set definition
idx = list(range(X_training.shape[0]))
val_idx = rng.sample(idx, int(X_training.shape[0]*0.25))
train_idx = [i for i in idx if i not in val_idx]
X_val  = X_training[val_idx]
X_training = X_training[train_idx]
Y_val  = Y_training[val_idx]
Y_training = Y_training[train_idx]

ratio = float(np.sum(Y_training == 0)) / np.sum(Y_training == 1)

In [ ]:
# Base parameters
base_params = {
    'boosting_type': 'gbdt',
    'objective': 'binary',
    'n_jobs': -1,
    'scale_pos_weight': ratio,
    'metric': 'auc',
    'verbosity': -1,  # Just to remove print of errors
    # Defaults for parameters not to be tested
    'n_estimators': 500,
    'learning_rate': 0.05,
    'max_depth': 5,
    'num_leaves': 31,
    'bagging_fraction': 0.8,  # Equivalent to subsample
    'bagging_freq': 5,        # Necessary when using bagging_fraction
    'feature_fraction': 0.8,  # Equivalent to colsample_bytree
    'min_gain_to_split': 0,   # Equivalent to gamma
    'lambda_l2': 1            # Equivalent to reg_lambda
}

# Parameters to analyze
params_to_test = {
    'min_gain_to_split': [0, 0.1, 0.5, 1, 2, 5],
    'lambda_l2': [0.1, 1, 10, 50, 100],
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'bagging_fraction': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'feature_fraction': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'max_depth': [3, 4, 5, 6, 8, 10],
    'num_leaves': [15, 31, 63, 127, 255],
    'n_estimators': [100, 300, 500, 800, 1000]
}

results = {}
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for param_name, param_values in params_to_test.items():
    print(f"Testing {param_name}...", end=" ")
    means = []
    stds = []

    for value in param_values:
        current_params = base_params.copy()
        current_params[param_name] = value

        # Use LGBMClassifier
        model = lgb.LGBMClassifier(**current_params)
        scores = cross_val_score(model, X_training, Y_training, cv=cv_strategy, scoring='roc_auc')

        means.append(scores.mean())
        stds.append(scores.std())

    results[param_name] = {
        'values': param_values,
        'means': means,
        'stds': stds
    }
    print("Done.")

# --- PLOTTING SECTION (Same logic as yours) ---
n_params = len(params_to_test)
n_cols = 3
n_rows = math.ceil(n_params / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
fig.suptitle('LightGBM Hyperparameter Performance Analysis (5-Fold CV)', fontsize=20)
axes = axes.flatten()

for i, (param_name, data) in enumerate(results.items()):
    ax = axes[i]
    x_vals = [str(v) for v in data['values']] # Convert to string for better x-axis spacing
    y_means = data['means']
    y_errs = data['stds']

    ax.errorbar(x_vals, y_means, yerr=y_errs, fmt='-o', capsize=5,
                color='forestgreen', ecolor='lightgray', elinewidth=2, markeredgecolor='darkgreen')

    ax.set_title(f'Effect of {param_name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Parameter Value', fontsize=10)
    ax.set_ylabel('Mean ROC AUC', fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.7)

    max_idx = np.argmax(y_means)
    ax.annotate(f'Max: {y_means[max_idx]:.4f}',
                xy=(max_idx, y_means[max_idx]),
                xytext=(0, 10), textcoords='offset points', ha='center',
                color='red', fontweight='bold')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# --- FINAL TRAINING ON BEST FOUND PARAMETERS ---
print("\nFINAL ANALYSIS: COMBINING BEST PARAMETERS")
best_found_params = base_params.copy()

for param_name, data in results.items():
    best_idx = np.argmax(data['means'])
    best_value = data['values'][best_idx]
    best_found_params[param_name] = best_value
    print(f"  - {param_name}: {best_value} (AUC: {data['means'][best_idx]:.4f})")

print("\nTraining final LightGBM model...")
final_model = lgb.LGBMClassifier(**best_found_params)
final_model.fit(X_training, Y_training)

# Evaluation
y_pred_final = final_model.predict(X_test)
y_prob_final = final_model.predict_proba(X_test)[:, 1]

auc_final = roc_auc_score(Y_test, y_prob_final)
acc_final = accuracy_score(Y_test, y_pred_final)

print(f"\nFinal Test ROC AUC:  {auc_final:.4f}")
print(f"Final Test Accuracy: {acc_final:.4f}")